# 🧬 Epigenetic Aging & Metadata Comparison on Cayo Santiago Macaque Blood Data

This notebook implements a complete pipeline to analyze DNA methylation levels from 572 rhesus macaques (*Macaca mulatta*) from the Cayo Santiago cohort. 
We load count matrices, calculate percent methylation, and compute four global statistics—**Mean**, **Median**, **Variability** (standard deviation), and **Shannon's Entropy**. Using these statistics, we build an unsupervised age-proxy model (mapped to true ages up to 28), rank individuals by relative health within their cohort, project the feature space using **UMAP**, and train/evaluate a supervised age predictor with 1000 runs of 5-fold cross-validation.

## 1. Feature Extraction (with Shannon's Entropy) & Epigenetic Windows Correlation
We load the high-dimensional Cayo Santiago counts data, compute the percent methylation matrix, and calculate global Mean, Median, Variability, and Shannon's Entropy for all 572 samples in R. We also perform Pearson correlation (EWAS) between each individual genomic window and the updated global Aging Score vector.

In [ ]:
import subprocess
import os
import pandas as pd
import numpy as np
import umap
import plotly.express as px
import plotly.graph_objects as go

# Define paths
rdata_path = '/Users/hellyr/Documents/windowCountData/CayoWindow161KCountsAndMetadata.RData'
features_csv = 'cayo_features.csv'
correlations_csv = 'cayo_top_correlations.csv'

r_code = f"""
load('{rdata_path}')

# Convert counts to percent methylation (fractional)
perc <- as.matrix(meth_counts / total_counts)

# Extract global features
print('Calculating mean, median, sd...')
mean_vals <- colMeans(perc, na.rm=TRUE)
median_vals <- apply(perc, 2, median, na.rm=TRUE)
sd_vals <- apply(perc, 2, sd, na.rm=TRUE)

print('Calculating Shannon entropy...')
entropy_func <- function(p) {{
  h <- -p * log2(p) - (1-p) * log2(1-p)
  h[p == 0 | p == 1] <- 0
  colMeans(h, na.rm=TRUE)
}}
entropy_vals <- entropy_func(perc)

df <- data.frame(
  sample = colnames(perc),
  mean = mean_vals,
  median = median_vals,
  variability = sd_vals,
  entropy = entropy_vals
)

# Standardize features to compute Aging Score
sd_std <- (df$variability - mean(df$variability)) / sd(df$variability)
entropy_std <- (df$entropy - mean(df$entropy)) / sd(df$entropy)
aging_score <- sd_std + entropy_std

# Compute Pearson correlation for each window with Aging Score
print('Computing window correlations (EWAS)...')
cor_vals <- cor(t(perc), aging_score, use = 'pairwise.complete.obs')
cor_df <- data.frame(
  window = rownames(cor_vals),
  correlation = cor_vals[, 1]
)
cor_df <- cor_df[!is.na(cor_df$correlation), ]

# Get top 10 hypermethylated and top 10 hypomethylated windows
cor_df_sorted <- cor_df[order(cor_df$correlation, decreasing = TRUE), ]
top_pos <- head(cor_df_sorted, 10)
top_neg <- tail(cor_df_sorted, 10)
top_cors <- rbind(top_pos, top_neg)

write.csv(df, '{features_csv}', row.names=FALSE)
write.csv(top_cors, '{correlations_csv}', row.names=FALSE)
print('Features and top correlations successfully exported!')
"""

# Write R script and run it
with open('extract_cayo_features.R', 'w') as f:
    f.write(r_code)

print("Running feature extraction in R...")
result = subprocess.run(['Rscript', 'extract_cayo_features.R'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("Error in R execution:", result.stderr)
    raise RuntimeError(f"R script failed: {{result.stderr}}")

# Clean up temporary R script
if os.path.exists('extract_cayo_features.R'):
    os.remove('extract_cayo_features.R')

# Load extracted features into pandas
df = pd.read_csv(features_csv)
df.head()

: 

## 2. Epigenetic Age Imputation (Regenerated Aging Score)
We standardize **Variability (SD)** and **Shannon's Entropy**, compute the updated **Aging Score** as their sum, and map the score uniformly to chronological age [3, 28] years.

In [ ]:
# Standardize features
df['mean_std'] = (df['mean'] - df['mean'].mean()) / df['mean'].std()
df['median_std'] = (df['median'] - df['median'].mean()) / df['median'].std()
df['variability_std'] = (df['variability'] - df['variability'].mean()) / df['variability'].std()
df['entropy_std'] = (df['entropy'] - df['entropy'].mean()) / df['entropy'].std()

# Calculate Aging Score as the sum of variability and entropy (both increase with biological aging)
df['aging_score'] = df['variability_std'] + df['entropy_std']

# Map ranks to chronological age [3, 28]
df = df.sort_values('aging_score').reset_index(drop=True)
N = len(df)
df['estimated_age'] = 3 + (28 - 3) * (df.index) / (N - 1)
df.head()

## 3. Epigenetic Health Ranking (Cohort-Wide)

We calculate the health rank across the entire cohort based on the unsupervised `aging_score` (Entropy + Variability). The lower the aging score, the healthier, with rank 1 being the healthiest individual.

In [ ]:
# Health score across all individuals (lower aging score = healthier = rank 1 is healthiest)
df['relative_health_rank'] = df['aging_score'].rank(method='min').astype(int)
df_sorted = df.sort_values('relative_health_rank').reset_index(drop=True)
df_sorted.head(10)


## 4. Dimensionality Reduction (UMAP of Epigenetic Disorder)

We project the 2D feature space of standard epigenetic disorder (Variability and Entropy) into a 2D UMAP space to visualize the epigenetic aging trajectory.

In [ ]:
# Scale features for UMAP
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
features_umap = ['variability', 'entropy']
scaled_features = scaler.fit_transform(df_sorted[features_umap])

# Fit UMAP in 2D
reducer = umap.UMAP(n_components=2, random_state=42)
embedding = reducer.fit_transform(scaled_features)

df_sorted['umap_1'] = embedding[:, 0]
df_sorted['umap_2'] = embedding[:, 1]
df_sorted.head()


## 5. Interactive UMAP Visualization

We plot the UMAP coordinates, coloring each sample by its unsupervised estimated age.

In [ ]:
fig = px.scatter(
    df_sorted,
    x='umap_1',
    y='umap_2',
    color='estimated_age',
    title='UMAP Projection of Cayo Santiago Epigenetic Aging Profiles (Variability & Entropy)',
    labels={'umap_1': 'UMAP 1', 'umap_2': 'UMAP 2', 'estimated_age': 'Unsupervised Age'},
    hover_name='sample',
    hover_data={
        'umap_1': False,
        'umap_2': False,
        'estimated_age': ':.2f',
        'relative_health_rank': True,
        'variability': ':.4f',
        'entropy': ':.4f'
    },
    color_continuous_scale='plasma'
)

fig.update_traces(
    marker=dict(size=8, opacity=0.8, line=dict(width=0.5, color='DarkSlateGrey')),
    selector=dict(mode='markers')
)

fig.update_layout(
    template='plotly_dark',
    title_font=dict(size=18, family='Outfit, sans-serif'),
    paper_bgcolor='rgba(15, 15, 15, 1)',
    plot_bgcolor='rgba(15, 15, 15, 1)',
    margin=dict(l=40, r=40, t=60, b=40),
    width=900,
    height=600
)

fig.show()


## 6. Annotation of Most Predictive Epigenetic Windows
We load the Pearson correlation coefficients computed in Step 1 to visualize the top 20 individual genomic windows (top 10 hypermethylated and top 10 hypomethylated) that correlate most strongly with the epigenetic aging score.

In [ ]:
# Load top correlations
cor_df = pd.read_csv(correlations_csv)
cor_df = cor_df.sort_values('correlation', ascending=True)

# Create a category color column
cor_df['type'] = np.where(cor_df['correlation'] > 0, 'Hypermethylated (Age-gain)', 'Hypomethylated (Age-loss)')

fig_bar = px.bar(
    cor_df,
    x='correlation',
    y='window',
    color='type',
    orientation='h',
    title='Top 20 Genomic Windows Most Predictive of Epigenetic Aging & Health (Cayo Santiago)',
    labels={'correlation': 'Pearson Correlation with Aging Score', 'window': 'Genomic Window ID'},
    color_discrete_map={
        'Hypermethylated (Age-gain)': '#E06666',
        'Hypomethylated (Age-loss)': '#6FA8DC'
    }
)

fig_bar.update_layout(
    template='plotly_dark',
    title_font=dict(size=18, family='Outfit, sans-serif'),
    xaxis=dict(showgrid=True, gridcolor='rgba(255, 255, 255, 0.1)', range=[-1.05, 1.05]),
    yaxis=dict(showgrid=False, title_font=dict(size=14)),
    paper_bgcolor='rgba(15, 15, 15, 1)',
    plot_bgcolor='rgba(15, 15, 15, 1)',
    margin=dict(l=120, r=40, t=60, b=40),
    width=900,
    height=600,
    legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center')
)

fig_bar.show()

## 7. Cohort Summary & Top Health Rankings

Here we show the top 10 overall healthiest individuals in the cohort according to the epigenetic aging score.

In [ ]:
top_health = df_sorted.head(10)
top_health_summary = top_health[['relative_health_rank', 'sample', 'estimated_age', 'variability', 'entropy']]
top_health_summary


## 8. Supervised Clocks Evaluation (Grouped 5-Fold Cross-Validation)

We evaluate three models predicting chronological age from sequence and methylation information:
1. **Base Elastic Net Clock**: Ridge regression on 216 selected features.
2. **CpGPT Clock**: Ridge regression on 1025-dimensional sequence embeddings (2001bp context + CpG density feature) from the Nucleotide Transformer.
3. **MuLan-Methyl Clock**: Ridge regression on 768-dimensional sequence embeddings (101bp context) from MuLan-Methyl BERT.

In [ ]:
# Load model cross-validation predictions
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

model_preds = pd.read_csv('cayo_model_predictions_v2.csv')

# Display cross-validation results table
def print_metrics(y_true, y_pred, name):
    r, _ = pearsonr(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name}:")
    print(f"  Pearson r: {r:.4f}")
    print(f"  R2: {r2:.4f}")
    print(f"  MAE: {mae:.4f} years")
    print(f"  RMSE: {rmse:.4f} years\n")

print_metrics(model_preds['true_age'], model_preds['en_predicted_age'], "Base Elastic Net Clock (216 selected features)")
print_metrics(model_preds['true_age'], model_preds['cpgpt_density_predicted_age'], "CpGPT Clock (2001bp context + CpG density feature)")
print_metrics(model_preds['true_age'], model_preds['mulan_predicted_age'], "MuLan-Methyl Clock (101bp context)")


## 9. Predictions vs. Chronological Age Visualizations

We plot true chronological age vs. predictions/estimations for all four models side-by-side.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

merged_df = pd.read_csv('cayo_final_predictions.csv')

def plot_pred_vs_true(y_col, title):
    fig = px.scatter(
        merged_df,
        x='true_age',
        y=y_col,
        color='true_age',
        labels={'true_age': 'Chronological Age (Years)', y_col: 'Predicted/Estimated Age (Years)'},
        title=title,
        hover_name='sample',
        color_continuous_scale='plasma'
    )
    
    # Add identity line
    min_val = min(merged_df['true_age'].min(), merged_df[y_col].min())
    max_val = max(merged_df['true_age'].max(), merged_df[y_col].max())
    fig.add_shape(
        type="line", x0=min_val, y0=min_val, x1=max_val, y1=max_val,
        line=dict(color="rgba(255, 255, 255, 0.4)", width=2, dash="dash")
    )
    
    # Add OLS regression trendline
    m, b = np.polyfit(merged_df['true_age'], merged_df[y_col], 1)
    x_range = np.linspace(merged_df['true_age'].min(), merged_df['true_age'].max(), 100)
    fig.add_trace(
        go.Scatter(
            x=x_range, y=m*x_range + b,
            name="OLS Fit", line=dict(color="white", width=2)
        )
    )
    
    fig.update_layout(
        template='plotly_dark',
        paper_bgcolor='rgba(15, 15, 15, 1)',
        plot_bgcolor='rgba(15, 15, 15, 1)',
        width=450,
        height=450,
        title_font=dict(size=14, family='Outfit, sans-serif'),
        coloraxis_showscale=False
    )
    fig.show()

plot_pred_vs_true('unsupervised_estimated_age', 'True Age vs. Unsupervised Estimated Age (Entropy + Variability)')
plot_pred_vs_true('en_predicted_age', 'True Age vs. Base Elastic Net CV Predicted Age')
plot_pred_vs_true('cpgpt_density_predicted_age', 'True Age vs. CpGPT + CpG Density CV Predicted Age')

merged_df['model_diff'] = merged_df['en_predicted_age'] - merged_df['cpgpt_density_predicted_age']
fig = px.scatter(
    merged_df,
    x='true_age',
    y='model_diff',
    color='true_age',
    labels={'true_age': 'Chronological Age (Years)', 'model_diff': 'EN Pred Age - CpGPT Pred Age'},
    title='Difference between Elastic Net and CpGPT Predicted Age',
    hover_name='sample',
    color_continuous_scale='plasma'
)
fig.add_shape(
    type="line", x0=merged_df['true_age'].min(), y0=0, x1=merged_df['true_age'].max(), y1=0,
    line=dict(color="rgba(255, 255, 255, 0.4)", width=2, dash="dash")
)
fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='rgba(15, 15, 15, 1)',
    plot_bgcolor='rgba(15, 15, 15, 1)',
    width=600,
    height=450,
    title_font=dict(size=14, family='Outfit, sans-serif'),
    coloraxis_showscale=False
)
fig.show()
